[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/corrections/seance2_correction.ipynb)

# Séance 3.2 — Comparer deux groupes — hasard ou vrai écart ?

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- expliquer pourquoi une moyenne calculée sur un échantillon n'est jamais exacte
- construire un intervalle de confiance à 95 % par rééchantillonnage
- comparer deux groupes avec un test t et lire sa p-value
- distinguer « pas de différence » de « pas de différence détectable »
- repérer les deux pièges qui rendent un test faux : dépendance et tests répétés

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

fr = cmd.query("pays == 'France'")["ca"]      ## 253 paniers francais
de = cmd.query("pays == 'Allemagne'")["ca"]   ## 250 paniers allemands
print(len(fr), "commandes francaises,", len(de), "allemandes")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Deux marchés à comparer

> **Votre mission :**
> - Extraire les paniers belges dans `be` et espagnols dans `es`.
> - Afficher les effectifs et les moyennes ; mettre la moyenne belge arrondie à 2 décimales dans `moy_be`.

In [ ]:
be = cmd.query("pays == 'Belgique'")["ca"]   ## 72 commandes
es = cmd.query("pays == 'Espagne'")["ca"]    ## 64 commandes

moy_be = round(be.mean(), 2)   ## 446,70 euros
print(len(be), "commandes belges, moyenne", moy_be)
print(len(es), "commandes espagnoles, moyenne", round(es.mean(), 2))

In [ ]:
verifier("1 - moyenne belge", moy_be == 446.7,
         "guillemets doubles a l'exterieur, simples autour du nom du pays")

### Exercice 2 — L'intervalle de confiance belge

> **Votre mission :**
> - Rééchantillonner `be` 1000 fois (avec remise, `random_state=i`) → `boot_be`.
> - En tirer les bornes de l'intervalle à 95 % → `bas_be` et `haut_be` (arrondies à 2 décimales).

In [ ]:
# replace=True : tirage avec remise, sinon on retire toujours
# exactement les memes commandes et la moyenne ne bouge jamais
boot_be = pd.Series([be.sample(len(be), replace=True, random_state=i).mean()
                     for i in range(1000)])   ## 1 000 belgiques possibles

# 95 % centraux : on coupe 2,5 % en bas et 2,5 % en haut
bas_be = round(boot_be.quantile(0.025), 2)    ## borne basse
haut_be = round(boot_be.quantile(0.975), 2)   ## borne haute
print("panier belge : entre", bas_be, "et", haut_be)

In [ ]:
verifier("2a - borne basse", bas_be == 384.58, "quantile(0.025)")
verifier("2b - borne haute", haut_be == 521.95,
         "quantile(0.975), et replace=True dans le sample")

### Exercice 3 — Belgique contre Espagne

> **Votre mission :**
> - Tester l'écart entre `be` et `es` → `p_be_es` (arrondie à 4 décimales).
> - Conclut-on à un écart au seuil de 5 % ? Mettre `True` ou `False` dans `conclut`.

In [ ]:
# equal_var=False : on ne suppose pas la meme dispersion des deux cotes
p_be_es = round(stats.ttest_ind(be, es, equal_var=False).pvalue, 4)

conclut = p_be_es < 0.05   ## 0,0683 : juste au-dessus du seuil
print("p =", p_be_es, "| on conclut a un ecart :", conclut)

# 0,068 : juste au-dessus du seuil. L'ecart de 157 EUR est important,
# mais 72 et 64 commandes ne suffisent pas a l'etablir. Ce n'est pas
# "pas de difference" : c'est "pas assez de donnees pour le dire".

In [ ]:
verifier("3a - p-value Belgique/Espagne", p_be_es == 0.0683,
         "stats.ttest_ind(be, es, equal_var=False).pvalue")
verifier("3b - conclusion au seuil de 5 %", not conclut,
         "0,0683 est-il inferieur a 0,05 ?")

### Exercice 4 — Le prix d'un petit effectif

> **Votre mission :**
> - Le Japon affiche le panier moyen le plus élevé du fichier. Sur combien de commandes ?
> - Calculer la **largeur** de son intervalle de confiance à 95 % → `larg_jp` (arrondie à 2 décimales).
> - Comparez-la à la largeur française (184,29 €).

In [ ]:
jp = cmd.query("pays == 'Japon'")["ca"]
bj = pd.Series([jp.sample(len(jp), replace=True, random_state=i).mean()
                for i in range(1000)])

larg_jp = round(bj.quantile(0.975) - bj.quantile(0.025), 2)   ## la largeur
print(len(jp), "commandes | largeur de l'intervalle :", larg_jp, "euros")

# 2 193 EUR de largeur contre 184 pour la France : sur 12 commandes,
# la moyenne n'est plus une mesure, c'est une impression.

In [ ]:
verifier("4 - largeur de l'intervalle japonais", larg_jp == 2193.18,
         "quantile(0.975) moins quantile(0.025)")

### Exercice 5 — Compter les individus, pas les lignes

> **Votre mission :**
> - Combien de clients distincts derrière les commandes irlandaises ? → `cli_irl`
> - Et derrière les britanniques ? → `cli_uk`

In [ ]:
# nunique compte les valeurs DISTINCTES, count compterait les lignes
cli_irl = cmd.query("pays == 'Irlande'")["client_id"].nunique()      ## 2
cli_uk = cmd.query("pays == 'Royaume-Uni'")["client_id"].nunique()   ## 235

print(cli_irl, "clients irlandais |", cli_uk, "clients britanniques")

In [ ]:
verifier("5a - clients irlandais", cli_irl == 2, "nunique() et non count()")
verifier("5b - clients britanniques", cli_uk == 235, "nunique() sur client_id")

### Exercice 6 — La taille de l'effet

> **Votre mission :**
> - Calculer l'écart en euros entre le panier irlandais et le britannique → `ecart_eur` (arrondi à 2 décimales).
> - Et le rapport entre les deux → `rapport` (arrondi à 2 décimales).
> - Un test dit si un écart existe ; ces deux nombres disent s'il compte.

In [ ]:
irl = cmd.query("pays == 'Irlande'")["ca"]
uk = cmd.query("pays == 'Royaume-Uni'")["ca"]

ecart_eur = round(irl.mean() - uk.mean(), 2)   ## en euros : parlant
rapport = round(irl.mean() / uk.mean(), 2)     ## en multiple : comparable

print(ecart_eur, "euros d'ecart |", rapport, "fois plus")

# A mettre systematiquement A COTE de la p-value : elle dit qu'un ecart
# existe, ces deux nombres disent s'il vaut une decision.

In [ ]:
verifier("6a - ecart en euros", ecart_eur == 626.46, "moyenne irlandaise moins britannique")
verifier("6b - rapport", rapport == 2.59, "divisez la moyenne irlandaise par la britannique")

### Exercice 7 — Vingt tests sur rien

> **Votre mission :**
> - Refaire les 20 comparaisons entre deux moitiés du fichier tirées au hasard.
> - Compter combien de p-values passent sous 0,05 → `nb_sig`.
> - Rappel : il n'y a **aucune** différence à trouver, les groupes sont tirés au hasard.

In [ ]:
ps = []
for i in range(20):
    a = cmd.sample(frac=0.5, random_state=i)
    b = cmd.drop(a.index)
    ps.append(stats.ttest_ind(a["ca"], b["ca"], equal_var=False).pvalue)

nb_sig = (pd.Series(ps) < 0.05).sum()   ## 1 sur 20, soit exactement 5 %
print(nb_sig, "test(s) significatif(s) sur 20, alors qu'il n'y a rien a trouver")

# Un sur vingt, soit exactement 5 % : c'est la definition du seuil,
# pas un accident. Tester tout contre tout garantit de "trouver".

In [ ]:
verifier("7 - faux positifs sur 20 tests", nb_sig == 1,
         "le seuil de significativite usuel est 0,05")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - On vous demande une recommandation écrite sur l'arbitrage France / Allemagne.
> - Calculer l'écart observé entre les deux moyennes → `ecart_fr_de` (arrondi à 2 décimales).
> - Récupérer la p-value du test → `p_fr_de` (arrondie à 3 décimales).
> - Puis rédigez votre recommandation en commentaire, en trois phrases maximum.

In [ ]:
ecart_fr_de = round(de.mean() - fr.mean(), 2)   ## la taille de l'effet
p_fr_de = round(stats.ttest_ind(fr, de, equal_var=False).pvalue, 3)   ## le test

print("ecart :", ecart_fr_de, "euros | p =", p_fr_de)

# Recommandation possible :
# "L'ecart de panier moyen entre l'Allemagne et la France est de 10,71 EUR,
#  soit 2 % — et il n'est pas mesurable sur nos volumes actuels (p = 0,87).
#  Ces donnees ne permettent pas d'arbitrer entre les deux marches : il faut
#  trancher sur le cout d'acquisition ou la marge, pas sur le panier."

In [ ]:
verifier("8a - ecart France/Allemagne", ecart_fr_de == 10.71,
         "moyenne allemande moins moyenne francaise")
verifier("8b - p-value", p_fr_de == 0.874, "arrondissez a 3 decimales")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — Un intervalle de confiance sur la médiane

> **Votre mission :**
> - Le rééchantillonnage ne sert pas qu'aux moyennes. Construire l'intervalle de confiance à 95 % de la **médiane** britannique.
> - Comparer sa largeur à celle de l'intervalle de la moyenne, calculé sur les mêmes commandes.
> - Lequel des deux est le plus étroit, et pourquoi ?

In [ ]:
uk = cmd.query("pays == 'Royaume-Uni'")["ca"]

med = pd.Series([uk.sample(len(uk), replace=True, random_state=i).median()
                 for i in range(1000)])   ## meme methode, autre statistique
moy = pd.Series([uk.sample(len(uk), replace=True, random_state=i).mean()
                 for i in range(1000)])

print("mediane :", round(med.quantile(0.025), 2), "-", round(med.quantile(0.975), 2))
print("moyenne :", round(moy.quantile(0.025), 2), "-", round(moy.quantile(0.975), 2))

# L'intervalle de la mediane est nettement plus etroit : elle ne bouge
# pas quand un gros panier entre ou sort du tirage. La robustesse vue
# en seance 3.1 se paie ici en precision gagnee.

### Question 10 — Le jeudi est-il vraiment meilleur ?

> **Votre mission :**
> - Le jeudi concentre le plus de commandes. Les paniers y sont-ils aussi plus élevés que le reste de la semaine ?
> - Comparer les paniers du jeudi à ceux de tous les autres jours réunis.
> - Attention à la formulation de votre conclusion.

In [ ]:
jeudi = cmd.query("jour == 'jeudi'")["ca"]    ## le jour le plus charge
autres = cmd.query("jour != 'jeudi'")["ca"]   ## != : tout le reste

print(round(jeudi.mean(), 2), "contre", round(autres.mean(), 2))
print("p =", round(stats.ttest_ind(jeudi, autres, equal_var=False).pvalue, 4))

# p = 0,578 : le jeudi rassemble plus de commandes, mais rien ne permet
# de dire qu'elles sont plus GROSSES. Deux questions differentes, deux
# reponses differentes — et un rapport qui les confondrait serait faux.

### Question 11 — Un test qui ne suppose pas de moyenne

> **Votre mission :**
> - Le test t compare des **moyennes** — or vous savez depuis la séance 3.1 que la moyenne décrit mal ces paniers.
> - Refaire la comparaison France / Allemagne avec un test de **Mann-Whitney**, qui raisonne sur les rangs.
> - Les deux tests concluent-ils pareil ?
> - *Nouveau :* `stats.mannwhitneyu(a, b)`.

In [ ]:
print("test t        : p =", round(stats.ttest_ind(fr, de, equal_var=False).pvalue, 3))
print("Mann-Whitney  : p =", round(stats.mannwhitneyu(fr, de).pvalue, 3))

# Meme conclusion : on ne peut pas departager les deux marches. Quand
# deux tests bati sur des hypotheses differentes s'accordent, la
# conclusion est solide. Quand ils divergent, c'est le signal qu'il faut
# regarder la distribution avant de choisir.

### Question 12 — Le test irlandais, au bon niveau

> **Votre mission :**
> - Le test Royaume-Uni / Irlande vu en cours traitait 256 commandes comme 256 observations indépendantes.
> - Refaire l'analyse **au niveau du client** : le panier moyen de chaque client irlandais, puis de chaque client britannique.
> - Combien d'observations reste-t-il du côté irlandais ? Que devient le test ?

In [ ]:
# Une ligne par CLIENT : c'est la bonne unite d'observation
par_client = cmd.groupby(["pays", "client_id"])["ca"].mean().reset_index()

irl_cli = par_client.query("pays == 'Irlande'")["ca"]
uk_cli = par_client.query("pays == 'Royaume-Uni'")["ca"]

print("clients irlandais   :", len(irl_cli), "->", irl_cli.round(2).tolist())
print("clients britanniques:", len(uk_cli))

# Deux observations. Un test statistique sur deux points ne veut rien
# dire, et leurs paniers moyens vont du simple au triple (716 et 2 134).
# Ce que le test de la partie 1 mesurait n'etait pas un ecart entre deux
# MARCHES, c'etait le comportement de deux acheteurs.

### Question 13 — Corriger le seuil quand on multiplie les tests

> **Votre mission :**
> - Comparer les dix principaux marchés deux à deux fait 45 tests. À 5 %, combien de « faux positifs » attendrait-on par pur hasard ?
> - La correction de **Bonferroni** consiste à diviser le seuil par le nombre de tests. Calculer le seuil corrigé.
> - Compter combien de comparaisons restent significatives avec l'ancien seuil, puis avec le nouveau.
> - *Nouveau :* `itertools.combinations(liste, 2)` énumère toutes les paires.

In [ ]:
from itertools import combinations

effectifs = cmd["pays"].value_counts()
gros = effectifs[effectifs >= 20].index

ps = [stats.ttest_ind(cmd.query("pays == @a")["ca"],
                      cmd.query("pays == @b")["ca"], equal_var=False).pvalue
      for a, b in combinations(gros, 2)]
ps = pd.Series(ps)

seuil = 0.05 / len(ps)   ## Bonferroni : le seuil divise par le nombre de tests
print(len(ps), "comparaisons | seuil corrige :", round(seuil, 5))
print("significatives a 0,05 :", (ps < 0.05).sum())
print("significatives apres correction :", (ps < seuil).sum())

# Le seuil passe de 5 % a 0,11 %. Les ecarts qui survivent a cette
# exigence sont ceux dont on peut parler ; les autres ne sont pas
# forcement faux, mais ces donnees ne permettent pas de les affirmer.

### Question 14 — L'effectif qu'il aurait fallu

> **Votre mission :**
> - L'écart France / Allemagne de 10,71 € n'est pas détectable sur 250 commandes de chaque côté. Combien en aurait-il fallu ?
> - Plutôt que de tâtonner, on peut calculer directement l'effectif nécessaire.
> - Il vaut environ `2 * (1.96 * ecart_type / ecart)**2` par groupe, où `ecart_type` est la dispersion typique des deux marchés.
> - Combien de commandes par pays aurait-il fallu ? Rapportez ce nombre aux 250 dont vous disposez.

In [ ]:
ecart = de.mean() - fr.mean()   ## ce qu'on veut detecter
ecart_type = np.sqrt((fr.std() ** 2 + de.std() ** 2) / 2)   ## la dispersion

n_requis = 2 * (1.96 * ecart_type / ecart) ** 2   ## par groupe

print("ecart observe :", round(ecart, 2), "euros")
print("dispersion    :", round(ecart_type, 1), "euros")
print("il aurait fallu", int(n_requis), "commandes par pays")

# Environ 38 500 commandes par marche, contre 250 disponibles : 150 fois
# plus. Autant dire que la question "France ou Allemagne ?" ne se
# tranchera JAMAIS sur le panier moyen. Le savoir avant de lancer
# l'analyse fait gagner une semaine.

### Question 15 — La note de synthèse

> **Votre mission :**
> - Un dirigeant vous écrit : *« L'Irlande fait un panier 2,6 fois supérieur au Royaume-Uni et c'est statistiquement prouvé (p < 0,001). Répliquons le modèle irlandais partout. »*
> - Produire les trois chiffres qui montrent pourquoi cette phrase ne tient pas.
> - Puis rédigez votre réponse en commentaire, en trois phrases.

In [ ]:
irl_cmd = cmd.query("pays == 'Irlande'")

print("commandes irlandaises :", len(irl_cmd))                    ## 256
print("clients derriere      :", irl_cmd["client_id"].nunique())  ## 2
print("panier moyen de chacun:",
      irl_cmd.groupby("client_id")["ca"].mean().round(2).tolist())

# Reponse possible :
# "La p-value est calculee comme si nos 256 commandes irlandaises
#  venaient de 256 acheteurs independants. Elles viennent de DEUX
#  comptes, dont les paniers moyens vont de 716 a 2 134 EUR : il n'y a
#  pas de 'modele irlandais', il y a deux grossistes. Ce que ces donnees
#  etablissent, c'est une dependance a deux clients — un risque, pas une
#  recette a repliquer."